# Meqpy Tutorial — 5b. Molecule

[← Previous: 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5c. Molecule Examples →](05c_Molecule_Examples.ipynb)

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

### Overview

- [5. Spatial Resolution](#spatial)
    - 5.1 Cube class [→ 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb)
        - 5.1.1 Cubes from file
        - 5.1.2 Cubes from 2p<sub>z</sub> vector
    - 5.2 Transition Class [→ 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb)
        - 5.2.1 Dyson
            - Coordinates and Slice Height
            - Properties and Methods
    - [5.3 Molecule](#spatial_molecule)
        - [5.3.1 Charging Transitions: Dyson](#spatial_molecule_dyson)
            - [Add Dyson Transition](#spatial_molecule_dyson_add)
            - [Missing Dyson Transitions](#spatial_molecule_dyson_missing)
            - [Molecule and Dyson Shape](#spatial_molecule_dyson_shape)
            - [Dyson Amplitudes](#spatial_molecule_dyson_amplitudes)
        - [5.3.2 Helper Functions](#spatial_molecule_helper)
        - [5.3.3 Charging Rates](#spatial_molecule_charging_rates)
            - [Coupling to plane wave Sample](#spatial_molecule_charging_rates_normal)
            - [Coupling to *s*-wave Tip](#spatial_molecule_charging_rates_dyson)
            - [Point Spectroscopy](#spatial_molecule_charging_rates_pointspec)
    - 5.4 Example Experiments [→ 5c Molecule Examples](05c_Molecule_Examples.ipynb)
        - 5.4.1 Constant Height Map
        - 5.4.2 I(V) Point Spectroscopy


<a id='spatial'></a>
## 5. Spatial Resolution
Another extension of the ``System`` class is the ``Molecule`` class, which allows systems to be solved with spatial resolution. For this, molecular orbitals can be used to determine the spatial dependence of various transition rates — for example, Dyson orbitals for the charge transitions. These orbitals must be computed by external means, e.g. using DFT or tight-binding methods, and then loaded into ``meqpy`` via the ``Cube`` class. A ``Cube`` object can then be used to instantiate a ``Transition`` object, such as ``Dyson``, which handles the charging transition rates in a ``Molecule``.

In this chapter, we have so far discussed the ``Cube`` class and the ``Transition`` and ``Dyson`` classes and are now looking at the ``Molecule`` class.

<a id='spatial_molecule'></a>
### 5.3 Molecule

The ``Molecule`` class is the spatial extension of the ``System`` class, using ``Transition`` objects to determine the spatial dependence of various transition rates.

``Molecule`` can be instantiated in the same way as ``System``, with the addition of the following parameters:
- ``padding``: spatial ``Transition`` objects will be padded by ``padding`` points on each side, default is 0
- ``tip_radius``: tip radius in Ångstrom, required for ``Dyson`` transitions, default is 2Å

In [ ]:
molecule = meqpy.Molecule(
    hwhm=100e-3,
    reorg_shift=200e-3,
    tip_radius=2.0,
    padding=20,
)


molecule.states = [
    meqpy.State("GS", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("PIR", energy=1.5, charge=+1, multiplicity=2),
    meqpy.State("NIR", energy=1.0, charge=-1, multiplicity=2),
]

<a id='spatial_molecule_dyson'></a>
#### 5.3.1 Charging Transitions: Dyson
Dyson orbitals not only describe the spatial dependence of charging transitions, but also determine their overall amplitude. For closed-shell systems, the Dyson orbitals are well approximated by the canonical orbitals, e.g. HOMO and LUMO, with a total amplitude of $\braket{\psi|\psi} = 1$. For open-shell systems with multireference configurations, this assumption no longer holds and the amplitude of the Dyson orbitals may take any value between 0 and 1. This must be taken into account when calculating the charging rates, even if no spatial resolution is required.

<a id='spatial_molecule_dyson_add'></a>
##### Adding Dyson Transitions
``Dyson`` objects are bi-directional transitions between two states with a charge difference of one electron. To add them to a ``Molecule`` one needs to provide the two states, as well as the ``Dyson`` object itself.

There are two ways to do this:
1) use the ``Dyson.add_dyson(a, b, dyson)`` method, with ``a`` and ``b`` being two states in the system and ``dyson`` a ``Dyson`` object.
2) write directly into the ``Dyson.dyson_dict``, with tuples ``(a, b)`` being the keys and ``Dyson`` objects the values

The order of the states does not matter when adding the transitions. The keys will be sorted internally to avoid assigning two ``Dyson`` objects to the same transition with keys ``(a, b)`` and ``(b, a)``.

In [ ]:
homo_dyson = meqpy.Dyson("./tutorial_files/naphthalene_homo.cub")
lumo_dyson = meqpy.Dyson("./tutorial_files/naphthalene_lumo.cub")

# states can be defined by labels or index
molecule.add_dyson("PIR", "GS", homo_dyson)
molecule.add_dyson(0, "NIR", lumo_dyson)

molecule.dyson_dict

In [ ]:
molecule.dyson_dict = {
    ("PIR", "GS"): homo_dyson,
    (0, "NIR"): lumo_dyson,
}

molecule.dyson_dict

<a id='spatial_molecule_dyson_missing'></a>
##### Missing Dyson Transitions
``Dyson`` transitions can be added to any combination of states with a charge difference of one electron and a multiplicity difference of 1. The latter criterion can be overruled by setting ``spin_selection_rule = False`` (see [2.3 Spin Selection Rule](02_System_and_States.ipynb)). All transitions without a Dyson orbital assigned to them will be considered forbidden and the transition rate is set to zero.

``Molecule.missing_dysons`` returns a list of all allowed transitions without a registered ``Dyson`` object.

In [ ]:
molecule.dyson_dict = {("GS", "PIR"): homo_dyson}
molecule.missing_dysons

In [ ]:
molecule.add_dyson("GS", "NIR", lumo_dyson)
molecule.missing_dysons

<a id='spatial_molecule_dyson_shape'></a>
##### Molecule and Dyson Shape
``Molecule.dyson_shape`` returns the shape of the spatial extension ``(nx, ny)`` including the padding. ``Molecule.shape`` includes, similar to ``System.shape``, the number of states as the last two dimensions: ``(nx, ny, num_states, num_states)``.

In [ ]:
# no padding -> shape of unpadded slice
molecule.padding = 0
molecule.dyson_shape

In [ ]:
# padding will change the dyson shape
molecule.padding = 20
molecule.dyson_shape

In [ ]:
# shape of transition + shape of system
molecule.shape

<a id='spatial_molecule_dyson_amplitudes'></a>
##### Dyson Amplitudes
The amplitudes of the Dyson orbitals can be obtained via the property ``Molecule.dyson_amplitudes``, which returns a symmetric matrix for all transitions within ``Molecule`` with the corresponding Dyson amplitude. Missing or forbidden transitions will be filled with 0.

In [ ]:
# add Dyson orbital with amplitude of 0.5
half_cube = meqpy.Cube("./tutorial_files/naphthalene_homo.cub")
half_cube.data *= 1 / np.sqrt(2)
half_dyson = meqpy.Dyson(half_cube)

molecule.add_dyson("GS", "PIR", half_dyson)

molecule.dyson_amplitudes

<a id='spatial_molecule_helper'></a>
#### 5.3.2 Helper Functions
The ``Molecule`` class provides the following helper functions:

Properties:
- ``spacing``: (2,2) matrix representing the pixel dimensions
- ``origin``: origin of the spatial data, excluding the padding
- ``x`` and ``y``: Cartesian coordinates of the x and y axes, including the padding set in ``Molecule.padding``
- ``mesh_cartesian``: returns a meshgrid of the xy-plane in Cartesian coordinates, including ``Molecule.padding``

Methods:
- ``get_xy_indices(points)``: returns the indices corresponding to the given coordinates of points in the xy-plane

``points`` can either be a tuple of two floats ``(x, y)`` or a list of such tuples.


In [ ]:
# get indices of single point in plane
points = (-7.0, 2.4)
molecule.get_xy_indices(points)

In [ ]:
# get indices for a list of points
points = [(-7.5, 2.4), (5.0, -1.5), (0.0, 6.8)]
molecule.get_xy_indices(points)

<a id='spatial_molecule_charging_rates'></a>
#### 5.3.3 Charging Rates
There are three different methods to calculate the charging rates in a ``Molecule`` object:
- without spatial resolution
- with spatial resolution, over the full grid
- with spatial resolution, at selected points

The term "spatial resolution" refers to charge transfer to or from an electrode with an $s$-wave type wave function, i.e. an STM tip. This can be described within the Tersoff-Hamann approximation and yields the charging rates as a function of the tip position. Coupling to a planar surface, on the other hand, does not introduce any spatial variation in the charging rate, as the planar surface is assumed to be isotropic. Nevertheless, the charging rates must still be rescaled by the Dyson orbital amplitude to capture the multireference nature of the charge transitions.

<a id='spatial_molecule_charging_rates_normal'></a>
The ``Molecule.charging_rates`` method is expanded with respect to the ``System`` class by multiplying the calculated charging rates by the ``Molecule.dyson_amplitudes`` matrix. This behavior can be disabled by using the parameter ``scale_by_dyson=False``.

In [ ]:
# instantiate Molecule
molecule = meqpy.Molecule(
    lineshape="dirac",
    kappa_mode="constant",
)

# add states
molecule.states = [
    meqpy.State("GS", energy=0.0, charge=0, multiplicity=1),
    meqpy.State("PIR", energy=1.0, charge=+1, multiplicity=2),
    meqpy.State("NIR", energy=1.0, charge=-1, multiplicity=2),
]

# load cube files
homo_cube = meqpy.Cube("./tutorial_files/naphthalene_homo.cub")
lumo_cube = meqpy.Cube("./tutorial_files/naphthalene_lumo.cub")

# manipulate homo_cube, as if the Dyson amplitude would be half
homo_cube.data *= 1 / np.sqrt(2)

# add Dyson transitions
molecule.add_dyson("GS", "PIR", meqpy.Dyson(homo_cube))
molecule.add_dyson("GS", "NIR", meqpy.Dyson(lumo_cube))

# get charging rates
molecule.charging_rates(z=5.0, bias=0.0)

<a id='spatial_molecule_charging_rates_dyson'></a>
The charging rates via an $s$-wave type electrode, i.e. an STM tip, can be calculated using the ``Molecule.charging_rates_dyson`` method.
The method uses the Tersoff-Hamann approach [[1]] and calculates the square of the Dyson orbital's wave function at the center of the tip for a given molecule-tip distance and all points in the xy-plane of the registered Dyson transitions.

To do so, the Dyson orbital is extrapolated exponentially to the height ``z + Molecule.tip_radius`` and squared. Using an estimation according to equation (10) in [[1]], the squared wave function is translated into a charging rate, considering the selected lineshape and the Clebsch-Gordan prefactors as well.

**Note**: The prefactor to translate the squared wave function into a charging rate is just an estimation for an arbitrary metallic lead. It is by no means precise and only gives an order-of-magnitude estimation of the current. 

[1]: https://doi.org/10.1103/PhysRevB.31.805

In [ ]:
spatial_charging_rates = molecule.charging_rates_dyson(z=5.0, bias=0.0)

spatial_charging_rates.shape

In [ ]:
# Plot spatial dependency of charging rates from PIR/NIR to GS

fig, ax = plt.subplots(1, 2, figsize=(5, 3))

vrange = np.max(np.abs(spatial_charging_rates))
ax[0].pcolormesh(
    molecule.y,
    molecule.x,
    spatial_charging_rates[..., 0, 1],
    cmap="grey",
    vmin=0,
    vmax=vrange,
)
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].axes.set_aspect("equal")
ax[0].set_title("PIR $\\rightarrow$ GS")

ax[1].pcolormesh(
    molecule.y,
    molecule.x,
    spatial_charging_rates[..., 0, 2],
    cmap="grey",
    vmin=0,
    vmax=vrange,
)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].axes.set_aspect("equal")
ax[1].set_title("NIR $\\rightarrow$ GS")

plt.tight_layout(pad=2)
plt.show()

Note: By default, the ``Molecule.charging_rates_dyson`` method will raise a warning if some valid charging transitions have no Dyson transition assigned. The warning can be turned off by setting ``warn_missing_dysons=False``. Missing transitions will be set to zero.

The method accepts 1D arrays as input for the ``z`` and ``bias`` parameters as well. This can lead to very large multidimensional arrays, possibly exceeding the memory limit of the hardware. The method will raise a warning, if the expected size of the array exceeds ``mem_warn_limit`` in GB, which is set to 4GB by default.

In [ ]:
# set mem_warn_limit to 100MB
z = np.arange(3, 8)
bias = np.linspace(-2, 2, 101)
molecule.charging_rates_dyson(z, bias, mem_warn_limit=0.1).shape

<a id='spatial_molecule_charging_rates_pointspec'></a>
In case only a few points in the xy-plane are required, the ``Molecule.charging_rates_pointspec`` method can be used to reduce the memory load. It is important to note, that the extrapolation of the wave function happens in reciprocal space and can only be done for the full xy-plane. ``charging_rates_pointspec`` runs a for-loop over all ``(z, bias)`` combinations, calculates the corresponding xy-plane from which it stores the points in question and discards the rest of the plane. This method is slower than ``charging_rates_dyson`` but reduces the required memory.

To select the necessary points, one can use the ``Molecule.get_xy_indices`` method.

In [ ]:
# choose points in cartesian coordinats and plot on the charging rates
points_xy = [
    (2.5, 2.2),  # x, y in Angstrom
    (4.2, 0.0),  # x, y in Angstrom
]

fig, ax = plt.subplots(1, 2, figsize=(5, 3))

vrange = np.max(np.abs(spatial_charging_rates))
ax[0].pcolormesh(
    molecule.y,
    molecule.x,
    spatial_charging_rates[..., 0, 1],
    cmap="grey",
    vmin=0,
    vmax=vrange,
)
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].axes.set_aspect("equal")
ax[0].set_title("PIR $\\rightarrow$ GS")

ax[1].pcolormesh(
    molecule.y,
    molecule.x,
    spatial_charging_rates[..., 0, 2],
    cmap="grey",
    vmin=0,
    vmax=vrange,
)
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].axes.set_aspect("equal")
ax[1].set_title("NIR $\\rightarrow$ GS")

for iax in ax:
    for point in points_xy:
        x, y = point
        iax.plot(y, x, "o")

plt.tight_layout(pad=2)
plt.show()

In [ ]:
points_idx = molecule.get_xy_indices(points_xy)

z = np.arange(3, 8)
bias = np.linspace(-2, 2, 101)
point_rates = molecule.charging_rates_pointspec(points_idx, z, bias)

# (num_points, num_z, num_bias, num_states, num_states)
point_rates.shape

---

[← Previous: 5a. Cubes and Transitions](05a_Cubes_and_Transitions.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5c. Molecule Examples →](05c_Molecule_Examples.ipynb)